# 13 最終戦略レポート(オフライン自己完結 HTML)

`quantkit.visualization.strategy_report` は、戦略+ベースライン+(任意で)IC・コストスイープを **ネット不要で開ける単一 HTML**(plotly.js を inline 埋め込み)にまとめる。勝者だけでなくベースラインと並べ、注意書き(合成/コスト・税は仮定/失敗も提示)を必ず載せる。

In [ ]:
import numpy as np
import pandas as pd
from quantkit import signals as S, backtest as B, visualization as V
from quantkit.features.price import cross_sectional_zscore

# Synthetic panel with a mild persistent edge (no network). Real data:
# quantkit.data.price_panel. Figures below work the same on real backtests.
rng = np.random.default_rng(11)
idx = pd.bdate_range('2015-01-01', periods=1500)
names = [f'A{i}' for i in range(10)]
edge = np.linspace(-0.0003, 0.0004, len(names))
rets = pd.DataFrame(
    {n: rng.normal(edge[i], 0.012, len(idx)) for i, n in enumerate(names)}, index=idx
)
prices = 100 * np.exp(rets.cumsum())
# momentum long-short, monthly rebalanced; vs equal-weight benchmark
sig = S.momentum_signal(prices, lookback=252, skip=21)
w = B.rebalanced(S.long_short_quantile(sig.lag(1).score, 0.3), 'ME')
strategy = B.run_backtest(w, rets, cost_model=B.CostModel(5, 2), lag=1)
equal_weight = B.buy_and_hold(rets)
results = {'momentum_LS': strategy, 'equal_weight': equal_weight}
print('built:', list(results), '| OOS days:', len(strategy.returns))

## IC とコストスイープ(レポートに添える)

In [ ]:
from quantkit.labels import forward_return
# feature-level IC: trailing momentum vs forward return (cross-sectional rank corr)
mom = sig.score  # standardized momentum score
fwd = forward_return(prices, 21)
df = pd.DataFrame({'p': mom.stack(future_stack=True), 'r': fwd.stack(future_stack=True)}).dropna()
ic_by_date = df.groupby(level=0).apply(
    lambda g: g['p'].corr(g['r'], method='spearman') if g['p'].nunique() > 2 else np.nan
)
ic = pd.Series({'momentum': float(ic_by_date.mean()), 'equal_weight': 0.0}, name='mean_IC')
rows = [{'cost_bps': b, 'sharpe': B.sharpe(B.run_backtest(w, rets, cost_model=B.CostModel(b, 0), lag=1), 252)}
        for b in [0, 5, 10, 20, 50]]
sweep = pd.DataFrame(rows).set_index('cost_bps')
print('mean IC (momentum):', round(ic['momentum'], 4))

## レポート生成 → `reports/strategy_report.html`

In [ ]:
from pathlib import Path
from quantkit.utils.paths import REPO_ROOT

out = Path(REPO_ROOT) / 'reports' / 'strategy_report.html'
path = V.strategy_report(
    results, out,
    title='IRP — Momentum L/S vs Equal-Weight (synthetic)',
    subtitle='Walk-forward research demo — not investment advice',
    ic=ic, cost_sweep=sweep, periods=252,
)
size_mb = path.stat().st_size / 1e6
print(f'wrote {path}  ({size_mb:.1f} MB, self-contained / offline)')
assert path.exists() and size_mb > 1.0  # inline plotly.js embedded

ブラウザでそのまま開ける(CDN 不要)。NB07 のモデル(Tier0-2)も同じ `results` 辞書に入れれば 同一レポートで **モデル vs ベースライン** を並置できる。

**正直な注記**: 合成データ・単純コスト・税/制約未考慮。*方法論*の実演であり投資結果ではない。